# 🩻 NIH Chest X-ray: แสดงภาพพร้อมตีกรอบรอยโรค (Bounding Box Visualization)

สมุดโน้ตบุ๊กนี้ใช้สำหรับ:
1. โหลดข้อมูลพิกัด Bounding Box จาก `BBox_List_2017.csv` (~984 รอยโรคจาก 880 ภาพ)
2. ดึงภาพเอกซเรย์ปอดต้นฉบับจากโฟลเดอร์ `images_001` ถึง `images_012`
3. ตีกรอบสี่เหลี่ยม (Bounding Box) และใส่ Label ชนิดของโรคอย่างชัดเจนบนภาพ
4. แสดงตัวอย่างภาพเดี่ยว, ภาพที่มีหลายรอยโรค (Multi-BBox) และภาพรวมทั้ง 8 ชนิดโรคที่มี Bounding Box

In [ ]:
import os
import glob
import random
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# ตรวจสอบ Path หลักของชุดข้อมูล
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'BBox_List_2017.csv').exists():
    DATASET_DIR = CURRENT_DIR
else:
    DATASET_DIR = CURRENT_DIR.parent

print(f"Dataset Directory: {DATASET_DIR}")
print("นำเข้าไลบรารีสำเร็จเรียบร้อย!")

In [ ]:
# 1. โหลดข้อมูล BBox_List_2017.csv
bbox_csv_path = DATASET_DIR / 'BBox_List_2017.csv'
df_bbox = pd.read_csv(bbox_csv_path).iloc[:, :6]
df_bbox.columns = ['Image Index', 'Finding Label', 'x', 'y', 'w', 'h']

print(f"จำนวนกรอบรอยโรคทั้งหมด: {len(df_bbox)} กล่อง")
print(f"จำนวนภาพที่มีกรอบรอยโรค: {df_bbox['Image Index'].nunique()} ภาพ\n")

print("--- สรุปจำนวนรอยโรคแต่ละชนิดในชุดข้อมูล BBox ---")
print(df_bbox['Finding Label'].value_counts())
df_bbox.head(10)

In [ ]:
# ทำ Indexing ค้นหา path ของภาพทั้งหมดล่วงหน้าเพื่อให้เข้าถึงได้ทันที (ใช้เวลาเพียง ~0.5 วินาที)
image_path_map = {}
for img_file in DATASET_DIR.glob('images_*/images/*.png'):
    image_path_map[img_file.name] = img_file

print(f"สแกนพบไฟล์ภาพในชุดข้อมูลทั้งหมด: {len(image_path_map):,} ภาพ")

In [ ]:
# กำหนดโทนสีเฉพาะสำหรับแต่ละโรคให้แยกแยะง่าย
DISEASE_COLORS = {
    'Atelectasis': '#FF3366',       # แดงชมพู
    'Cardiomegaly': '#FF9900',      # ส้ม
    'Effusion': '#00D4FF',          # ฟ้าสว่าง
    'Infiltration': '#00FF66',      # เขียวนีออน
    'Infiltrate': '#00FF66',        # เขียวนีออน
    'Mass': '#FF00FF',              # ม่วงมาเจนต้า
    'Nodule': '#FFFF00',            # เหลืองสด
    'Pneumonia': '#FF5722',         # ส้มแดง
    'Pneumothorax': '#76FF03'       # เขียวมะนาว
}

def visualize_image_bbox(image_name, df_boxes=df_bbox, figsize=(8, 8), show_info=True):
    """
    ฟังก์ชันแสดงภาพ X-ray พร้อมตีกรอบรอยโรคทั้งหมดที่พบในภาพนั้น
    """
    if image_name not in image_path_map:
        print(f"ไม่พบไฟล์ภาพ: {image_name}")
        return
        
    img_path = image_path_map[image_name]
    img = Image.open(img_path)
    
    # ดึงกรอบทั้งหมดของภาพนี้
    boxes = df_boxes[df_boxes['Image Index'] == image_name]
    
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img, cmap='gray')
    
    for _, row in boxes.iterrows():
        label = row['Finding Label']
        x, y, w, h = float(row['x']), float(row['y']), float(row['w']), float(row['h'])
        color = DISEASE_COLORS.get(label, '#FF0000')
        
        # ตีกรอบสี่เหลี่ยมรอบรอยโรค
        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=2.5,
            edgecolor=color,
            facecolor='none',
            linestyle='-'
        )
        ax.add_patch(rect)
        
        # ใส่ป้ายชื่อโรคพร้อมพื้นหลังอ่านง่าย
        ax.text(
            x, max(0, y - 10),
            f"{label} [{int(w)}x{int(h)}]",
            color='black',
            fontsize=10,
            weight='bold',
            bbox=dict(boxstyle='square,pad=0.2', facecolor=color, edgecolor='none', alpha=0.9)
        )
    
    findings = ', '.join(boxes['Finding Label'].unique())
    ax.set_title(f"Image: {image_name} | Finding(s): {findings}", fontsize=12, pad=12, weight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

### ตัวอย่างที่ 1: แสดงภาพเดี่ยวพร้อมตีกรอบรอยโรค (Single Image Example)
ลองเลือกภาพตัวอย่างแรก `00013118_008.png` ที่พบภาวะ *Atelectasis (ปอดแฟบ)*

In [ ]:
sample_img = df_bbox.iloc[0]['Image Index']
visualize_image_bbox(sample_img)

### ตัวอย่างที่ 2: ภาพที่มีรอยโรคหลายตำแหน่ง หรือหลายโรคพร้อมกัน (Multi-BBox Example)

In [ ]:
# ค้นหาภาพที่มี Bounding Box มากกว่า 1 กรอบ
multi_box_counts = df_bbox['Image Index'].value_counts()
multi_box_images = multi_box_counts[multi_box_counts > 1].index.tolist()
print(f"พบภาพที่มีหลายรอยโรค/หลายกรอบทั้งหมด: {len(multi_box_images)} ภาพ")

# แสดงตัวอย่างภาพที่มีมากกว่า 1 กรอบ
if multi_box_images:
    visualize_image_bbox(multi_box_images[0])

### ตัวอย่างที่ 3: รวมภาพตัวอย่างครบทั้ง 8 โรคที่มี Bounding Box
แสดงผลเปรียบเทียบรอยโรคแต่ละชนิดในรูปแบบ Grid 8 ช่อง

In [ ]:
diseases = df_bbox['Finding Label'].unique()
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, disease in enumerate(diseases):
    sample_row = df_bbox[df_bbox['Finding Label'] == disease].iloc[0]
    img_name = sample_row['Image Index']
    img_path = image_path_map[img_name]
    img = Image.open(img_path)
    
    ax = axes[i]
    ax.imshow(img, cmap='gray')
    
    color = DISEASE_COLORS.get(disease, 'red')
    x, y, w, h = sample_row['x'], sample_row['y'], sample_row['w'], sample_row['h']
    
    rect = patches.Rectangle((x, y), w, h, linewidth=2.5, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(
        x, max(0, y - 10), disease,
        color='black', fontsize=9, weight='bold',
        bbox=dict(boxstyle='square,pad=0.2', facecolor=color, edgecolor='none', alpha=0.9)
    )
    ax.set_title(f"{disease}\n({img_name})", fontsize=11, weight='bold', pad=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

### ตัวอย่างที่ 4: สุ่มดูภาพรอยโรค (Random Disease Explorer)
รันเซลล์นี้เพื่อสุ่มดูภาพรอยโรคอื่น ๆ เพิ่มเติม สามารถระบุชื่อโรคที่ต้องการดูเจาะจงได้ในพารามิเตอร์ `disease_name`

In [ ]:
def explore_random(disease_name=None):
    if disease_name:
        subset = df_bbox[df_bbox['Finding Label'].str.lower() == disease_name.lower()]
        if subset.empty:
            print(f"ไม่พบโรค '{disease_name}' ในชุด BBox. โรคที่มีได้แก่: {list(DISEASE_COLORS.keys())}")
            return
    else:
        subset = df_bbox
        
    random_row = subset.sample(1).iloc[0]
    visualize_image_bbox(random_row['Image Index'])

# สุ่มดูภาพรอยโรคแบบใดก็ได้ (หรือใส่ชื่อโรค เช่น explore_random('Cardiomegaly'))
explore_random()